In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc

In [2]:
torch.cuda.empty_cache()
gc.collect()

20

In [3]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [4]:
from cuml.preprocessing import MaxAbsScaler
from cuml.preprocessing import MinMaxScaler
from cuml.preprocessing import Normalizer
from cuml.preprocessing import RobustScaler
from cuml.preprocessing import StandardScaler
from cuml.preprocessing import Binarizer
from cuml.preprocessing import FunctionTransformer
from cuml.preprocessing import KBinsDiscretizer
import cupy as cp
import time

In [5]:
class Normalization(object):
    def __init__(self, dataset):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X = cp.array(self.dataset_numpy_array)


    def MaxAbsScaler(self):
        global maxAbsScaler_global

        transformer = MaxAbsScaler().fit(self.X)
        df_maxAbs_scaler_transform = transformer.transform(self.X)
        
        maxAbsScaler_global = cudf.DataFrame(df_maxAbs_scaler_transform)
    
    def MinMaxScaler(self):
        global MinMaxScaler_global

        transformer = MinMaxScaler().fit(self.X)
        df_MinMaxScaler_transform = transformer.transform(self.X)

        MinMaxScaler_global = cudf.DataFrame(df_MinMaxScaler_transform)

    def Normalizer(self):
        global normalizer_global

        transformer = Normalizer().fit(self.X)
        df_normalizer_transform = transformer.transform(self.X)

        normalizer_global = cudf.DataFrame(df_normalizer_transform)

    def RobustScaler(self):
        global robust_scaler_global

        transformer = RobustScaler().fit(self.X)
        df_robust_scaler_transform = transformer.transform(self.X)

        robust_scaler_global = cudf.DataFrame(df_robust_scaler_transform)

    def StandardScaler(self):
        global standard_scaler_global

        transformer = StandardScaler().fit(self.X)
        df_standard_scaler_transform = transformer.transform(self.X)

        standard_scaler_global = cudf.DataFrame(df_standard_scaler_transform)

    def Binarizer(self):
        global binarizer_global

        transformer = Binarizer().fit(self.X)
        df_binarizer_transform = transformer.transform(self.X)

        binarizer_global = cudf.DataFrame(df_binarizer_transform)

    def FunctionTransformer(self):
        global function_transformer_global

        transformer = FunctionTransformer(func=cp.log1p)
        df_function_transformer_transform = transformer.transform(self.X)

        function_transformer_global = cudf.DataFrame(df_function_transformer_transform)

    def KBinsDiscretizer(self):
        global KBinsDiscretizer_global

        transformer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform').fit(self.X)
        df_KBinsDiscretizer_transform = transformer.transform(self.X)

        KBinsDiscretizer_global = cudf.DataFrame(df_KBinsDiscretizer_transform)


    def main(self):
        st = time.time()
        self.MaxAbsScaler()
        self.MinMaxScaler()
        self.Normalizer()
        self.RobustScaler()
        self.StandardScaler()
        self.Binarizer()
        self.FunctionTransformer()
        self.KBinsDiscretizer()

        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')
        

In [6]:
norm = Normalization(df)

In [7]:
norm.main()

Execution time: 0.5342438220977783 seconds


In [8]:
maxAbsScaler_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.0,1.0,1.0,1.0,0.408163,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.6,0.500000,1.0,0.0,0.692308,0.666667,0.375
1,0.0,0.0,0.0,0.0,0.255102,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.6,0.0,0.000000,0.0,0.0,0.538462,1.000000,0.125
2,0.0,1.0,1.0,1.0,0.285714,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.000000,1.0,0.0,0.692308,0.666667,1.000
3,0.0,1.0,0.0,1.0,0.275510,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.4,0.0,0.000000,0.0,0.0,0.846154,0.500000,0.750
4,0.0,1.0,1.0,1.0,0.244898,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.4,0.1,0.000000,0.0,0.0,0.846154,0.833333,0.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,0.459184,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.6,0.0,0.166667,0.0,1.0,0.384615,1.000000,0.875
253676,0.0,1.0,1.0,1.0,0.183673,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.8,0.0,0.000000,1.0,0.0,0.846154,0.333333,0.500
253677,0.0,0.0,0.0,1.0,0.285714,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.2,0.0,0.000000,0.0,0.0,0.153846,0.833333,0.250
253678,0.0,1.0,0.0,1.0,0.234694,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.6,0.0,0.000000,0.0,1.0,0.538462,0.833333,0.125


In [9]:
MinMaxScaler_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.0,1.0,1.0,1.0,0.325581,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.00,0.6,0.500000,1.0,0.0,0.666667,0.6,0.285714
1,0.0,0.0,0.0,0.0,0.151163,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.50,0.0,0.000000,0.0,0.0,0.500000,1.0,0.000000
2,0.0,1.0,1.0,1.0,0.186047,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.00,1.0,1.000000,1.0,0.0,0.666667,0.6,1.000000
3,0.0,1.0,0.0,1.0,0.174419,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.25,0.0,0.000000,0.0,0.0,0.833333,0.4,0.714286
4,0.0,1.0,1.0,1.0,0.139535,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.25,0.1,0.000000,0.0,0.0,0.833333,0.8,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,0.383721,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.50,0.0,0.166667,0.0,1.0,0.333333,1.0,0.857143
253676,0.0,1.0,1.0,1.0,0.069767,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.75,0.0,0.000000,1.0,0.0,0.833333,0.2,0.428571
253677,0.0,0.0,0.0,1.0,0.186047,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.00,0.0,0.000000,0.0,0.0,0.083333,0.8,0.142857
253678,0.0,1.0,0.0,1.0,0.127907,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.50,0.0,0.000000,0.0,1.0,0.500000,0.8,0.000000


In [10]:
normalizer_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.00000,0.020911,0.020911,0.020911,0.836425,0.020911,0.0,0.000000,0.000000,0.000000,...,0.020911,0.000000,0.104553,0.376391,0.313659,0.020911,0.000000,0.188196,0.083642,0.062732
1,0.00000,0.000000,0.000000,0.000000,0.929760,0.037190,0.0,0.000000,0.037190,0.000000,...,0.000000,0.037190,0.111571,0.000000,0.000000,0.000000,0.000000,0.260333,0.223142,0.037190
2,0.00000,0.018976,0.018976,0.018976,0.531337,0.000000,0.0,0.000000,0.000000,0.018976,...,0.018976,0.018976,0.094882,0.569290,0.569290,0.018976,0.000000,0.170787,0.075905,0.151811
3,0.00000,0.033241,0.000000,0.033241,0.897510,0.000000,0.0,0.000000,0.033241,0.033241,...,0.033241,0.000000,0.066482,0.000000,0.000000,0.000000,0.000000,0.365652,0.099723,0.199447
4,0.00000,0.036322,0.036322,0.036322,0.871719,0.000000,0.0,0.000000,0.036322,0.036322,...,0.036322,0.000000,0.072643,0.108965,0.000000,0.000000,0.000000,0.399538,0.181608,0.145287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.00000,0.021437,0.021437,0.021437,0.964680,0.000000,0.0,0.000000,0.000000,0.021437,...,0.021437,0.000000,0.064312,0.000000,0.107187,0.000000,0.021437,0.107187,0.128624,0.150061
253676,0.00000,0.045175,0.045175,0.045175,0.813157,0.000000,0.0,0.090351,0.000000,0.000000,...,0.045175,0.000000,0.180702,0.000000,0.000000,0.045175,0.000000,0.496929,0.090351,0.180702
253677,0.00000,0.000000,0.000000,0.034879,0.976612,0.000000,0.0,0.000000,0.034879,0.034879,...,0.034879,0.000000,0.034879,0.000000,0.000000,0.000000,0.000000,0.069758,0.174395,0.069758
253678,0.00000,0.040193,0.000000,0.040193,0.924448,0.000000,0.0,0.000000,0.000000,0.040193,...,0.040193,0.000000,0.120580,0.000000,0.000000,0.000000,0.040193,0.281354,0.200967,0.040193


In [11]:
robust_scaler_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.0,1.0,1.0,0.0,1.857143,1.0,0.0,0.0,-1.0,-1.0,...,0.0,0.0,3.0,9.0,5.000000,1.0,0.0,0.25,-0.5,-1.333333
1,0.0,0.0,0.0,-1.0,-0.285714,1.0,0.0,0.0,0.0,-1.0,...,-1.0,1.0,1.0,0.0,0.000000,0.0,0.0,-0.25,0.5,-2.000000
2,0.0,1.0,1.0,0.0,0.142857,0.0,0.0,0.0,-1.0,0.0,...,0.0,1.0,3.0,15.0,10.000000,1.0,0.0,0.25,-0.5,0.333333
3,0.0,1.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.75,-1.0,-0.333333
4,0.0,1.0,1.0,0.0,-0.428571,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.5,0.000000,0.0,0.0,0.75,0.0,-1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,0.0,2.571429,0.0,0.0,0.0,-1.0,0.0,...,0.0,0.0,1.0,0.0,1.666667,0.0,1.0,-0.75,0.5,0.000000
253676,0.0,1.0,1.0,0.0,-1.285714,0.0,0.0,2.0,-1.0,-1.0,...,0.0,0.0,2.0,0.0,0.000000,1.0,0.0,0.75,-1.5,-1.000000
253677,0.0,0.0,0.0,0.0,0.142857,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.0,0.0,0.000000,0.0,0.0,-1.50,0.0,-1.666667
253678,0.0,1.0,0.0,0.0,-0.571429,0.0,0.0,0.0,-1.0,0.0,...,0.0,0.0,1.0,0.0,0.000000,0.0,1.0,-0.25,0.0,-2.000000


In [12]:
standard_scaler_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,-0.322458,1.153688,1.165254,0.196922,1.757936,1.120927,-0.205637,-0.425292,-1.762814,-1.316872,...,0.226863,-0.303173,2.329121,1.998592,1.233999,2.223615,-0.887021,0.316900,-1.065595,-1.474487
1,-0.322458,-0.866785,-0.858182,-5.078164,-0.511806,1.120927,-0.205637,-0.425292,0.567275,-1.316872,...,-4.407954,3.298445,0.457294,-0.429630,-0.486592,-0.449718,-0.887021,-0.337933,0.963272,-2.440138
2,-0.322458,1.153688,1.165254,0.196922,-0.057858,-0.892119,-0.205637,-0.425292,-1.762814,0.759375,...,0.226863,3.298445,2.329121,3.617407,2.954590,2.223615,-0.887021,0.316900,-1.065595,0.939638
3,-0.322458,1.153688,-0.858182,0.196922,-0.209174,-0.892119,-0.205637,-0.425292,0.567275,0.759375,...,0.226863,-0.303173,-0.478619,-0.429630,-0.486592,-0.449718,-0.887021,0.971733,-2.080028,-0.026012
4,-0.322458,1.153688,1.165254,0.196922,-0.663122,-0.892119,-0.205637,-0.425292,0.567275,0.759375,...,0.226863,-0.303173,-0.478619,-0.024926,-0.486592,-0.449718,-0.887021,0.971733,-0.051162,-0.991662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,-0.322458,1.153688,1.165254,0.196922,2.514516,-0.892119,-0.205637,-0.425292,-1.762814,0.759375,...,0.226863,-0.303173,0.457294,-0.429630,0.086938,-0.449718,1.127369,-0.992766,0.963272,0.456813
253676,-0.322458,1.153688,1.165254,0.196922,-1.571019,-0.892119,-0.205637,2.439387,-1.762814,-1.316872,...,0.226863,-0.303173,1.393207,-0.429630,-0.486592,2.223615,-0.887021,0.971733,-3.094461,-0.991662
253677,-0.322458,-0.866785,-0.858182,0.196922,-0.057858,-0.892119,-0.205637,-0.425292,0.567275,0.759375,...,0.226863,-0.303173,-1.414532,-0.429630,-0.486592,-0.449718,-0.887021,-1.975015,-0.051162,-1.957312
253678,-0.322458,1.153688,-0.858182,0.196922,-0.814438,-0.892119,-0.205637,-0.425292,-1.762814,0.759375,...,0.226863,-0.303173,0.457294,-0.429630,-0.486592,-0.449718,1.127369,-0.337933,-0.051162,-2.440138


In [13]:
binarizer_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
3,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0
253676,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0
253677,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
253678,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0


In [14]:
function_transformer_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.000000,0.693147,0.693147,0.693147,3.713572,0.693147,0.0,0.000000,0.000000,0.000000,...,0.693147,0.000000,1.791759,2.944439,2.772589,0.693147,0.000000,2.302585,1.609438,1.386294
1,0.000000,0.000000,0.000000,0.000000,3.258097,0.693147,0.0,0.000000,0.693147,0.000000,...,0.000000,0.693147,1.386294,0.000000,0.000000,0.000000,0.000000,2.079442,1.945910,0.693147
2,0.000000,0.693147,0.693147,0.693147,3.367296,0.000000,0.0,0.000000,0.000000,0.693147,...,0.693147,0.693147,1.791759,3.433987,3.433987,0.693147,0.000000,2.302585,1.609438,2.197225
3,0.000000,0.693147,0.000000,0.693147,3.332205,0.000000,0.0,0.000000,0.693147,0.693147,...,0.693147,0.000000,1.098612,0.000000,0.000000,0.000000,0.000000,2.484907,1.386294,1.945910
4,0.000000,0.693147,0.693147,0.693147,3.218876,0.000000,0.0,0.000000,0.693147,0.693147,...,0.693147,0.000000,1.098612,1.386294,0.000000,0.000000,0.000000,2.484907,1.791759,1.609438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.000000,0.693147,0.693147,0.693147,3.828641,0.000000,0.0,0.000000,0.000000,0.693147,...,0.693147,0.000000,1.386294,0.000000,1.791759,0.000000,0.693147,1.791759,1.945910,2.079442
253676,0.000000,0.693147,0.693147,0.693147,2.944439,0.000000,0.0,1.098612,0.000000,0.000000,...,0.693147,0.000000,1.609438,0.000000,0.000000,0.693147,0.000000,2.484907,1.098612,1.609438
253677,0.000000,0.000000,0.000000,0.693147,3.367296,0.000000,0.0,0.000000,0.693147,0.693147,...,0.693147,0.000000,0.693147,0.000000,0.000000,0.000000,0.000000,1.098612,1.791759,1.098612
253678,0.000000,0.693147,0.000000,0.693147,3.178054,0.000000,0.0,0.000000,0.000000,0.693147,...,0.693147,0.000000,1.386294,0.000000,0.000000,0.000000,0.693147,2.079442,1.791759,0.693147


In [15]:
KBinsDiscretizer_global

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0,2,2,2,0,2,0,0,0,0,...,2,0,2,1,1,2,0,2,1,0
1,0,0,0,0,0,2,0,0,2,0,...,0,2,1,0,0,0,0,1,2,0
2,0,2,2,2,0,0,0,0,0,2,...,2,2,2,2,2,2,0,2,1,2
3,0,2,0,2,0,0,0,0,2,2,...,2,0,0,0,0,0,0,2,1,2
4,0,2,2,2,0,0,0,0,2,2,...,2,0,0,0,0,0,0,2,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0,2,2,2,1,0,0,0,0,2,...,2,0,1,0,0,0,2,1,2,2
253676,0,2,2,2,0,0,0,2,0,0,...,2,0,2,0,0,2,0,2,0,1
253677,0,0,0,2,0,0,0,0,2,2,...,2,0,0,0,0,0,0,0,2,0
253678,0,2,0,2,0,0,0,0,0,2,...,2,0,1,0,0,0,2,1,2,0


In [16]:
class Data_Generation(object):

    def __init__(self, dataset, size, data_type):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.size = size
        self.data_type = data_type
        self.HighBP_nums = []
        self.HighChol_nums = []
        self.CholCheck_nums = []
        self.Smoker_nums = []
        self.Stroke_nums = []
        self.PhysActivity_nums = []
        self.Fruits_nums = []
        self.Veggies_nums= []
        self.HvyAlcoholConsump_nums = []
        self.AnyHealthcare_nums = []
        self.NoDocbcCost_nums = []
        self.DiffWalk_nums = []
        self.Sex_nums = []
        self.BMI_nums = []
        self.Diabetes_nums = []
        self.GenHlth_nums = []
        self.MentHlth_nums = []
        self.PhysHlth_nums = []
        self.Age_nums = []
        self.Education_nums = []
        self.Income_nums = []
        self.generated_data = []

    def generating_data(self):
        self.HighBP_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.HighChol_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.CholCheck_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Smoker_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Stroke_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.PhysActivity_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Fruits_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Veggies_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.HvyAlcoholConsump_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.AnyHealthcare_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.NoDocbcCost_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.DiffWalk_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.Sex_nums = cp.random.choice([0, 1], size=self.size).astype(self.data_type)
        self.BMI_nums = cp.random.choice(range(12, 99), size=self.size).astype(self.data_type)
        self.Diabetes_nums = cp.random.choice(range(0, 3), size = self.size).astype(self.data_type)
        self.GenHlth_nums = cp.random.choice(range(1, 6), size = self.size).astype(self.data_type)
        self.MentHlth_nums = cp.random.choice(range(0, 31), size = self.size).astype(self.data_type)
        self.PhysHlth_nums = cp.random.choice(range(0, 31), size = self.size).astype(self.data_type)
        self.Age_nums = cp.random.choice(range(1, 14), size = self.size).astype(self.data_type)
        self.Education_nums = cp.random.choice(range(1, 7), size = self.size).astype(self.data_type)
        self.Income_nums = cp.random.choice(range(1, 9), size = self.size).astype(self.data_type)
        
        
    def generating_dataframe(self):
        global generated_data_global

        self.generated_data = cudf.DataFrame({"HighBP": self.HighBP_nums, 
                                            'HighChol': self.HighChol_nums,
                                            'CholCheck' : self.CholCheck_nums,
                                            'BMI' : self.BMI_nums, 
                                            'Smoker' : self.Smoker_nums, 
                                            'Stroke' : self.Stroke_nums, 
                                            'Diabetes' : self.Diabetes_nums,
                                            'PhysActivity' : self.PhysActivity_nums, 
                                            'Fruits' : self.Fruits_nums, 
                                            'Veggies' : self.Veggies_nums,
                                            'HvyAlcoholConsump' : self.HvyAlcoholConsump_nums, 
                                            'AnyHealthcare' : self.AnyHealthcare_nums, 
                                            'NoDocbcCost' : self.NoDocbcCost_nums,
                                            'GenHlth' : self.GenHlth_nums, 
                                            'MentHlth' : self.MentHlth_nums, 
                                            'PhysHlth' : self.PhysHlth_nums, 
                                            'DiffWalk' : self.DiffWalk_nums, 
                                            'Sex' : self.Sex_nums, 
                                            'Age' : self.Age_nums, 
                                            'Education' : self.Education_nums,
                                            'Income' : self.Income_nums})

        
        generated_data_global = self.generated_data

    def main(self):
        self.generating_data()
        self.generating_dataframe()

In [17]:
torch.cuda.empty_cache()
gc.collect()

20

In [18]:
gen_df = Data_Generation(df, 2000000, float)

In [19]:
torch.cuda.empty_cache
gc.collect()

0

In [20]:
gen_df.main()

In [21]:
generated_data_global

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,Veggies,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,0.0,15.0,0.0,0.0,2.0,0.0,1.0,1.0,...,1.0,0.0,4.0,9.0,10.0,1.0,0.0,11.0,5.0,6.0
1,0.0,0.0,1.0,34.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,5.0,26.0,1.0,0.0,0.0,5.0,2.0,2.0
2,0.0,1.0,1.0,16.0,1.0,1.0,2.0,0.0,1.0,1.0,...,1.0,0.0,3.0,26.0,16.0,0.0,0.0,12.0,1.0,6.0
3,1.0,0.0,1.0,28.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,5.0,18.0,1.0,0.0,0.0,12.0,4.0,6.0
4,0.0,1.0,0.0,38.0,1.0,1.0,1.0,0.0,0.0,0.0,...,1.0,1.0,3.0,14.0,29.0,1.0,1.0,9.0,6.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,1.0,0.0,1.0,16.0,0.0,1.0,2.0,0.0,1.0,0.0,...,0.0,0.0,3.0,5.0,26.0,1.0,1.0,10.0,2.0,2.0
1999996,1.0,0.0,0.0,51.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,4.0,19.0,27.0,1.0,0.0,12.0,3.0,3.0
1999997,0.0,0.0,0.0,15.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,4.0,17.0,26.0,1.0,0.0,8.0,1.0,4.0
1999998,1.0,1.0,1.0,22.0,1.0,1.0,2.0,1.0,0.0,1.0,...,1.0,0.0,3.0,22.0,16.0,1.0,0.0,7.0,5.0,6.0


In [22]:
norm_gen_data = Normalization(generated_data_global)

In [23]:
torch.cuda.empty_cache()
gc.collect()

0

In [24]:
norm_gen_data.main()

Execution time: 1.726557731628418 seconds


In [25]:
maxAbsScaler_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.0,0.0,0.0,0.153061,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.8,0.300000,0.333333,1.0,0.0,0.846154,0.833333,0.750
1,0.0,0.0,1.0,0.346939,0.0,0.0,0.5,0.0,0.0,1.0,...,1.0,1.0,1.0,0.866667,0.033333,0.0,0.0,0.384615,0.333333,0.250
2,0.0,1.0,1.0,0.163265,1.0,1.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.6,0.866667,0.533333,0.0,0.0,0.923077,0.166667,0.750
3,1.0,0.0,1.0,0.285714,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,0.600000,0.033333,0.0,0.0,0.923077,0.666667,0.750
4,0.0,1.0,0.0,0.387755,1.0,1.0,0.5,0.0,0.0,0.0,...,1.0,1.0,0.6,0.466667,0.966667,1.0,1.0,0.692308,1.000000,0.875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,1.0,0.0,1.0,0.163265,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.6,0.166667,0.866667,1.0,1.0,0.769231,0.333333,0.250
1999996,1.0,0.0,0.0,0.520408,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.8,0.633333,0.900000,1.0,0.0,0.923077,0.500000,0.375
1999997,0.0,0.0,0.0,0.153061,1.0,1.0,0.5,1.0,1.0,1.0,...,1.0,1.0,0.8,0.566667,0.866667,1.0,0.0,0.615385,0.166667,0.500
1999998,1.0,1.0,1.0,0.224490,1.0,1.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.6,0.733333,0.533333,1.0,0.0,0.538462,0.833333,0.750


In [26]:
MinMaxScaler_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.0,0.0,0.0,0.034884,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.75,0.300000,0.333333,1.0,0.0,0.833333,0.8,0.714286
1,0.0,0.0,1.0,0.255814,0.0,0.0,0.5,0.0,0.0,1.0,...,1.0,1.0,1.00,0.866667,0.033333,0.0,0.0,0.333333,0.2,0.142857
2,0.0,1.0,1.0,0.046512,1.0,1.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.50,0.866667,0.533333,0.0,0.0,0.916667,0.0,0.714286
3,1.0,0.0,1.0,0.186047,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.00,0.600000,0.033333,0.0,0.0,0.916667,0.6,0.714286
4,0.0,1.0,0.0,0.302326,1.0,1.0,0.5,0.0,0.0,0.0,...,1.0,1.0,0.50,0.466667,0.966667,1.0,1.0,0.666667,1.0,0.857143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,1.0,0.0,1.0,0.046512,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.50,0.166667,0.866667,1.0,1.0,0.750000,0.2,0.142857
1999996,1.0,0.0,0.0,0.453488,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.75,0.633333,0.900000,1.0,0.0,0.916667,0.4,0.285714
1999997,0.0,0.0,0.0,0.034884,1.0,1.0,0.5,1.0,1.0,1.0,...,1.0,1.0,0.75,0.566667,0.866667,1.0,0.0,0.583333,0.0,0.428571
1999998,1.0,1.0,1.0,0.116279,1.0,1.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.50,0.733333,0.533333,1.0,0.0,0.500000,0.8,0.714286


In [27]:
normalizer_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.000000,0.000000,0.000000,0.605844,0.000000,0.000000,0.080779,0.000000,0.040390,0.040390,...,0.040390,0.000000,0.161558,0.363507,0.403896,0.040390,0.000000,0.444286,0.201948,0.242338
1,0.000000,0.000000,0.022966,0.780836,0.000000,0.000000,0.022966,0.000000,0.000000,0.022966,...,0.022966,0.022966,0.114829,0.597110,0.022966,0.000000,0.000000,0.114829,0.045932,0.045932
2,0.000000,0.026822,0.026822,0.429153,0.026822,0.026822,0.053644,0.000000,0.026822,0.026822,...,0.026822,0.000000,0.080466,0.697374,0.429153,0.000000,0.000000,0.321865,0.026822,0.160933
3,0.027359,0.000000,0.027359,0.766046,0.027359,0.000000,0.000000,0.000000,0.027359,0.000000,...,0.027359,0.027359,0.136794,0.492458,0.027359,0.000000,0.000000,0.328305,0.109435,0.164153
4,0.000000,0.019375,0.000000,0.736235,0.019375,0.019375,0.019375,0.000000,0.000000,0.000000,...,0.019375,0.019375,0.058124,0.271244,0.561864,0.019375,0.019375,0.174371,0.116248,0.135622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,0.030373,0.000000,0.030373,0.485965,0.000000,0.030373,0.060746,0.000000,0.030373,0.000000,...,0.000000,0.000000,0.091119,0.151864,0.789694,0.030373,0.030373,0.303728,0.060746,0.060746
1999996,0.016066,0.000000,0.000000,0.819389,0.000000,0.000000,0.000000,0.016066,0.016066,0.000000,...,0.000000,0.000000,0.064266,0.305263,0.433794,0.016066,0.000000,0.192798,0.048199,0.048199
1999997,0.000000,0.000000,0.000000,0.416506,0.027767,0.027767,0.027767,0.027767,0.027767,0.027767,...,0.027767,0.027767,0.111068,0.472040,0.721944,0.027767,0.000000,0.222137,0.027767,0.111068
1999998,0.027146,0.027146,0.027146,0.597218,0.027146,0.027146,0.054293,0.027146,0.000000,0.027146,...,0.027146,0.000000,0.081439,0.597218,0.434340,0.027146,0.000000,0.190024,0.135731,0.162878


In [28]:
robust_scaler_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,-1.0,0.0,0.0,-0.909091,-1.0,0.0,0.5,-1.0,0.0,0.0,...,0.0,0.0,0.5,-0.3750,-0.3125,1.0,0.0,0.666667,0.666667,0.50
1,-1.0,0.0,1.0,-0.477273,-1.0,0.0,0.0,-1.0,-1.0,0.0,...,0.0,1.0,1.0,0.6875,-0.8750,0.0,0.0,-0.333333,-0.333333,-0.50
2,-1.0,1.0,1.0,-0.886364,0.0,1.0,0.5,-1.0,0.0,0.0,...,0.0,0.0,0.0,0.6875,0.0625,0.0,0.0,0.833333,-0.666667,0.50
3,0.0,0.0,1.0,-0.613636,0.0,0.0,-0.5,-1.0,0.0,-1.0,...,0.0,1.0,1.0,0.1875,-0.8750,0.0,0.0,0.833333,0.333333,0.50
4,-1.0,1.0,0.0,-0.386364,0.0,1.0,0.0,-1.0,-1.0,-1.0,...,0.0,1.0,0.0,-0.0625,0.8750,1.0,1.0,0.333333,1.000000,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,0.0,0.0,1.0,-0.886364,-1.0,1.0,0.5,-1.0,0.0,-1.0,...,-1.0,0.0,0.0,-0.6250,0.6875,1.0,1.0,0.500000,-0.333333,-0.50
1999996,0.0,0.0,0.0,-0.090909,-1.0,0.0,-0.5,0.0,0.0,-1.0,...,-1.0,0.0,0.5,0.2500,0.7500,1.0,0.0,0.833333,0.000000,-0.25
1999997,-1.0,0.0,0.0,-0.909091,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.5,0.1250,0.6875,1.0,0.0,0.166667,-0.666667,0.00
1999998,0.0,1.0,1.0,-0.750000,0.0,1.0,0.5,0.0,-1.0,0.0,...,0.0,0.0,0.0,0.4375,0.0625,1.0,0.0,0.000000,0.666667,0.50


In [29]:
standard_scaler_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,-1.000663,-0.998664,-0.998484,-1.591739,-1.000103,-0.999391,1.224124,-1.000764,0.999756,0.99957,...,0.999426,-0.999083,0.707756,-0.671478,-0.559716,1.001593,-0.99986,1.068758,0.878798,0.655677
1,-1.000663,-0.998664,1.001518,-0.835444,-1.000103,-0.999391,-0.000527,-1.000764,-1.000244,0.99957,...,0.999426,1.000917,1.415229,1.229508,-1.566739,-0.998409,-0.99986,-0.535213,-0.878431,-1.089788
2,-1.000663,1.001338,1.001518,-1.551934,0.999897,1.000609,1.224124,-1.000764,0.999756,0.99957,...,0.999426,-0.999083,0.000282,1.229508,0.111633,-0.998409,-0.99986,1.336086,-1.464174,0.655677
3,0.999337,-0.998664,1.001518,-1.074274,0.999897,-0.999391,-1.225177,-1.000764,0.999756,-1.00043,...,0.999426,1.000917,1.415229,0.334926,-1.566739,-0.998409,-0.99986,1.336086,0.293055,0.655677
4,-1.000663,1.001338,-0.998484,-0.676224,0.999897,1.000609,-0.000527,-1.000764,-1.000244,-1.00043,...,0.999426,1.000917,0.000282,-0.112365,1.566222,1.001593,1.00014,0.534101,1.464541,1.092043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,0.999337,-0.998664,1.001518,-1.551934,-1.000103,1.000609,1.224124,-1.000764,0.999756,-1.00043,...,-1.000574,-0.999083,0.000282,-1.118769,1.230548,1.001593,1.00014,0.801429,-0.878431,-1.089788
1999996,0.999337,-0.998664,-0.998484,-0.158759,-1.000103,-0.999391,-1.225177,0.999236,0.999756,-1.00043,...,-1.000574,-0.999083,0.707756,0.446749,1.342439,1.001593,-0.99986,1.336086,-0.292688,-0.653421
1999997,-1.000663,-0.998664,-0.998484,-1.591739,0.999897,1.000609,-0.000527,0.999236,0.999756,0.99957,...,0.999426,1.000917,0.707756,0.223103,1.230548,1.001593,-0.99986,0.266772,-1.464174,-0.217055
1999998,0.999337,1.001338,1.001518,-1.313104,0.999897,1.000609,1.224124,0.999236,-1.000244,0.99957,...,0.999426,-0.999083,0.000282,0.782217,0.111633,1.001593,-0.99986,-0.000556,0.878798,0.655677


In [30]:
binarizer_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0
2,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,...,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0
3,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0
4,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1999996,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1999997,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1999998,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,...,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0


In [31]:
function_transformer_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.000000,0.000000,0.000000,2.772589,0.000000,0.000000,1.098612,0.000000,0.693147,0.693147,...,0.693147,0.000000,1.609438,2.302585,2.397895,0.693147,0.000000,2.484907,1.791759,1.945910
1,0.000000,0.000000,0.693147,3.555348,0.000000,0.000000,0.693147,0.000000,0.000000,0.693147,...,0.693147,0.693147,1.791759,3.295837,0.693147,0.000000,0.000000,1.791759,1.098612,1.098612
2,0.000000,0.693147,0.693147,2.833213,0.693147,0.693147,1.098612,0.000000,0.693147,0.693147,...,0.693147,0.000000,1.386294,3.295837,2.833213,0.000000,0.000000,2.564949,0.693147,1.945910
3,0.693147,0.000000,0.693147,3.367296,0.693147,0.000000,0.000000,0.000000,0.693147,0.000000,...,0.693147,0.693147,1.791759,2.944439,0.693147,0.000000,0.000000,2.564949,1.609438,1.945910
4,0.000000,0.693147,0.000000,3.663562,0.693147,0.693147,0.693147,0.000000,0.000000,0.000000,...,0.693147,0.693147,1.386294,2.708050,3.401197,0.693147,0.693147,2.302585,1.945910,2.079442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,0.693147,0.000000,0.693147,2.833213,0.000000,0.693147,1.098612,0.000000,0.693147,0.000000,...,0.000000,0.000000,1.386294,1.791759,3.295837,0.693147,0.693147,2.397895,1.098612,1.098612
1999996,0.693147,0.000000,0.000000,3.951244,0.000000,0.000000,0.000000,0.693147,0.693147,0.000000,...,0.000000,0.000000,1.609438,2.995732,3.332205,0.693147,0.000000,2.564949,1.386294,1.386294
1999997,0.000000,0.000000,0.000000,2.772589,0.693147,0.693147,0.693147,0.693147,0.693147,0.693147,...,0.693147,0.693147,1.609438,2.890372,3.295837,0.693147,0.000000,2.197225,0.693147,1.609438
1999998,0.693147,0.693147,0.693147,3.135494,0.693147,0.693147,1.098612,0.693147,0.000000,0.693147,...,0.693147,0.000000,1.386294,3.135494,2.833213,0.693147,0.000000,2.079442,1.791759,1.945910


In [32]:
KBinsDiscretizer_global

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0,0,0,0,0,0,2,0,2,2,...,2,0,2,0,1,2,0,2,2,2
1,0,0,2,0,0,0,1,0,0,2,...,2,2,2,2,0,0,0,1,0,0
2,0,2,2,0,2,2,2,0,2,2,...,2,0,1,2,1,0,0,2,0,2
3,2,0,2,0,2,0,0,0,2,0,...,2,2,2,1,0,0,0,2,1,2
4,0,2,0,0,2,2,1,0,0,0,...,2,2,1,1,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,2,0,2,0,0,2,2,0,2,0,...,0,0,1,0,2,2,2,2,0,0
1999996,2,0,0,1,0,0,0,2,2,0,...,0,0,2,1,2,2,0,2,1,0
1999997,0,0,0,0,2,2,1,2,2,2,...,2,2,2,1,2,2,0,1,0,1
1999998,2,2,2,0,2,2,2,2,0,2,...,2,0,1,2,1,2,0,1,2,2


In [33]:
from cuml import LinearRegression
from cuml.linear_model import LinearRegression

In [34]:
lr = LinearRegression(fit_intercept = True, algorithm = "eig")
X = cudf.DataFrame(df.copy().drop(['HeartDiseaseorAttack'], axis = 1))
y = cudf.DataFrame(df['HeartDiseaseorAttack'].copy())

In [35]:
reg = lr.fit(X,y)
print(reg.coef_)

0     0.033506
1     0.038606
2     0.014458
3    -0.001249
4     0.023006
5     0.187874
6     0.023933
7     0.003166
8     0.003848
9     0.003523
10   -0.019211
11    0.008023
12    0.007689
13    0.032988
14   -0.000184
15    0.000954
16    0.049835
17    0.053135
18    0.011045
19    0.001019
20   -0.003531
dtype: float64


In [36]:
print(reg.intercept_)

-0.14540693374116526


In [37]:
torch.cuda.empty_cache
gc.collect()

0

In [38]:
X_gen = cudf.DataFrame(generated_data_global.copy())
preds = lr.predict(X_gen)
print(preds)

0          0.174980
1          0.081058
2          0.364427
3          0.188157
4          0.404079
             ...   
1999995    0.453406
1999996    0.141245
1999997    0.356433
1999998    0.389172
1999999    0.296674
Length: 2000000, dtype: float64


In [39]:
torch.cuda.empty_cache
gc.collect()

0

In [40]:
X = cudf.DataFrame(df.copy().drop(['HeartDiseaseorAttack'], axis = 1))
y = df['HeartDiseaseorAttack'].copy().to_numpy()
model = cuml.LogisticRegression().fit(X, y)
print(model.coef_)


         0         1         2         3         4         5        6   \
0  0.524871  0.613525  0.535572  0.001233  0.362968  0.983866  0.14569   

        7         8         9   ...        11        12        13        14  \
0  0.04078  0.005893  0.043384  ...  0.032207  0.261411  0.492163  0.002418   

         15       16        17        18        19        20  
0  0.001039  0.29217  0.759941  0.256252  0.012602 -0.043493  

[1 rows x 21 columns]


In [41]:
print(model.intercept_)

0   -7.987359
dtype: float64


In [42]:
torch.cuda.empty_cache
gc.collect()

0

In [43]:
X_gen = cudf.DataFrame(generated_data_global.copy())
preds = model.predict(X_gen)
print(preds)

0          0.0
1          0.0
2          0.0
3          0.0
4          0.0
          ... 
1999995    0.0
1999996    0.0
1999997    0.0
1999998    0.0
1999999    0.0
Length: 2000000, dtype: float64
